# Chronic Kidney Disease (CKD) Classification using Machine Learning
### Minor Project — Google Colab Notebook

**Dataset:** UCI Chronic Kidney Disease (`kidney_disease.csv`) — 400 patients, 24 clinical features + 1 target.

**Experiment 1 (this notebook's default): NO outlier removal.**
Experiment 2 (outlier capping) is at the end so you can compare.

**Evaluation:** 5-fold **Stratified** Cross-Validation → Confusion Matrix, TP/TN/FP/FN,
Accuracy, Sensitivity, Specificity, Precision, F1-score, ROC curve and AUC.

> **How to run this:** open [Google Colab](https://colab.research.google.com) →
> `File ▸ Upload notebook` → upload this `.ipynb` → then run each cell with `Shift + Enter`.
> Colab already has pandas, numpy, scikit-learn, matplotlib and seaborn installed,
> so there is nothing to `pip install`.


---
## Step 0 — Setup and uploading your dataset

Run the cell below and click **Choose Files** to upload `kidney_disease.csv` from your computer.

*(Alternative: put the file in Google Drive and mount it with
`from google.colab import drive; drive.mount('/content/drive')`.)*


In [ ]:
# --- Upload the CSV (Colab only). Skip this cell if running on your own PC. ---
try:
    from google.colab import files
    uploaded = files.upload()          # click "Choose Files" -> kidney_disease.csv
    CSV_PATH = list(uploaded.keys())[0]
except ImportError:
    CSV_PATH = "kidney_disease.csv"    # running locally: file sits next to the notebook

print("Using file:", CSV_PATH)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, roc_curve, auc, roc_auc_score)

RANDOM_STATE = 42     # fixes randomness so your results are reproducible
N_SPLITS     = 5      # 5-fold cross-validation

sns.set_theme(style="whitegrid")
print("Libraries loaded. pandas", pd.__version__, "| scikit-learn ok")


---
## Step 1 — Load the raw data and look at it honestly

Before cleaning anything, we look at what is actually in the file.
**Never trust a dataset you have not inspected.**


In [ ]:
raw = pd.read_csv(CSV_PATH)
print("Shape (rows, columns):", raw.shape)
raw.head()


In [ ]:
raw.info()


### What the 26 columns mean

| Column | Meaning | Type |
|---|---|---|
| `id` | Row number — **not a feature**, we drop it | drop |
| `age` | Age (years) | numeric |
| `bp` | Blood pressure (mm/Hg) | numeric |
| `sg` | Specific gravity of urine | numeric (ordered) |
| `al` | Albumin level (0–5) | numeric (ordered) |
| `su` | Sugar level (0–5) | numeric (ordered) |
| `bgr` | Blood glucose random (mgs/dl) | numeric |
| `bu` | Blood urea (mgs/dl) | numeric |
| `sc` | Serum creatinine (mgs/dl) | numeric |
| `sod` | Sodium (mEq/L) | numeric |
| `pot` | Potassium (mEq/L) | numeric |
| `hemo` | Haemoglobin (gms) | numeric |
| `pcv` | Packed cell volume | numeric |
| `wc` | White blood cell count (cells/cumm) | numeric |
| `rc` | Red blood cell count (millions/cmm) | numeric |
| `rbc` | Red blood cells (normal / abnormal) | categorical |
| `pc` | Pus cell (normal / abnormal) | categorical |
| `pcc` | Pus cell clumps (present / notpresent) | categorical |
| `ba` | Bacteria (present / notpresent) | categorical |
| `htn` | Hypertension (yes / no) | categorical |
| `dm` | Diabetes mellitus (yes / no) | categorical |
| `cad` | Coronary artery disease (yes / no) | categorical |
| `appet` | Appetite (good / poor) | categorical |
| `pe` | Pedal edema (yes / no) | categorical |
| `ane` | Anemia (yes / no) | categorical |
| **`classification`** | **ckd / notckd — this is the TARGET** | **target** |

**Target variable = `classification`.** We set **CKD = 1 (positive class)** and **notckd = 0**,
because in medical screening the disease is the event we want to *detect*.
This choice is what makes "Sensitivity" mean *"of all true CKD patients, how many did we catch?"*


---
## Step 2 — Find the dirty values (the part most students miss)

Three columns (`pcv`, `wc`, `rc`) look numeric but pandas loaded them as **text**.
Something non-numeric is hiding in them. Let's find it.


In [ ]:
# Print every distinct value of the text columns, using repr() so that
# invisible characters like tabs (\t) and spaces become VISIBLE.
raw_str = pd.read_csv(CSV_PATH, dtype=str)
for col in ["rbc","pc","pcc","ba","htn","dm","cad","appet","pe","ane",
            "classification","pcv","wc","rc"]:
    vals = sorted(repr(v) for v in raw_str[col].dropna().unique())
    show = vals if len(vals) <= 8 else vals[:4] + ["..."] + vals[-2:]
    print(f"{col:15s} ({raw_str[col].nunique()} unique): {show}")


### What you should see in the output above

Four real problems in this dataset:

1. **`classification`** has **3** values, not 2: `'ckd'`, `'notckd'` and **`'ckd\t'`**
   (a stray *tab* character). Left alone, `'ckd\t'` becomes a **third class** and breaks everything.
2. **`dm`** has **5** values instead of 2: `'yes'`, `'no'`, `'\tyes'`, `'\tno'`, `' yes'`.
3. **`cad`** has **3** values instead of 2: `'yes'`, `'no'`, `'\tno'`.
4. **`pcv`, `wc`, `rc`** contain the string **`'\t?'`** — a *missing-value placeholder*
   written as text. That single value forces the whole column to be text instead of numbers.

This is why they were not read as numeric. We fix all four in the next step.


---
## Step 3 — Data preprocessing (structural cleaning)

**Important idea about data leakage:** the cleaning below is **structural** — it fixes typos,
strips tab characters, converts text to numbers and maps the labels.
It uses **no information from the data distribution** (no mean, no median, no min/max).

Because it learns nothing statistical, it is **safe** to do on the whole dataset before splitting.

Anything that *does* learn from the data — **imputation (median/mode), scaling, outlier
fences, one-hot categories** — must happen **inside** cross-validation. We do that in Step 6.


In [ ]:
NUMERIC = ["age","bp","sg","al","su","bgr","bu","sc","sod","pot","hemo","pcv","wc","rc"]
CATEGORICAL = ["rbc","pc","pcc","ba","htn","dm","cad","appet","pe","ane"]
TARGET = "classification"

def load_and_clean(path):
    df = pd.read_csv(path)
    df = df.drop(columns=["id"])                       # (1) id is not a feature

    # (2) strip stray tabs/spaces from every text cell
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype) == "str":
            df[c] = df[c].astype("string").str.strip()

    # (3) '?' and empty string are MISSING values, not categories
    df = df.replace({"?": pd.NA, "": pd.NA})

    # (4) force pcv/wc/rc (and all numeric cols) to real numbers.
    #     errors="coerce" turns anything still unparseable into NaN.
    for c in NUMERIC:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # (5) lowercase categories so '\tNo' and 'no' merge into one category.
    #     .astype(object) keeps scikit-learn's imputer happy (it dislikes pd.NA).
    for c in CATEGORICAL:
        df[c] = (df[c].astype("string").str.lower()
                   .astype(object).where(lambda s: s.notna(), np.nan))

    # (6) TARGET: 'ckd\t' -> 'ckd'; CKD = 1 (positive), notckd = 0
    df[TARGET] = df[TARGET].astype("string").str.strip().str.lower()
    df[TARGET] = df[TARGET].map({"ckd": 1, "notckd": 0})
    assert df[TARGET].notna().all(), "A label failed to map - check spelling!"
    df[TARGET] = df[TARGET].astype(int)
    return df

df = load_and_clean(CSV_PATH)
X = df[NUMERIC + CATEGORICAL]
y = df[TARGET]

print("Clean shape:", df.shape)
print("\nDtypes are now correct (pcv/wc/rc are float64):")
print(df[["pcv","wc","rc"]].dtypes)
print("\nTarget now has exactly 2 classes:")
print(y.value_counts().rename({1:"ckd (1)", 0:"notckd (0)"}))
print("\ndm categories after cleaning:", sorted(df['dm'].dropna().unique()))
print("cad categories after cleaning:", sorted(df['cad'].dropna().unique()))


---
## Step 4 — Explore: class balance, missing values, distributions

These plots go straight into your report.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

counts = y.value_counts().sort_index()
ax[0].bar(["notckd (0)", "ckd (1)"], counts.values, color=["#2e86c1", "#c0392b"])
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 4, f"{v}\n({v/len(y)*100:.1f}%)", ha="center")
ax[0].set_title("Class distribution"); ax[0].set_ylabel("Number of patients")
ax[0].set_ylim(0, max(counts.values) * 1.25)

miss = (X.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
ax[1].barh(miss.index[::-1], miss.values[::-1], color="#c0392b")
ax[1].set_xlabel("% missing"); ax[1].set_title("Missing values per feature")

plt.tight_layout(); plt.show()

print(f"Total rows: {len(df)}")
print(f"Rows with at least one missing value: {X.isna().any(axis=1).sum()} "
      f"({X.isna().any(axis=1).mean()*100:.1f}%)")
print(f"Complete rows (no missing at all):   {(~X.isna().any(axis=1)).sum()}")


### Why we must **impute** and must **not** just delete rows

Look at the two numbers printed above. Only a small fraction of the 400 rows are complete.
If you used `df.dropna()` you would throw away the large majority of your dataset —
and the rows you throw away are **not random**: sick patients get more lab tests ordered,
so missingness itself is related to the outcome. Deleting them would bias the study.

So: **impute** (fill in) the missing values instead. Rule of thumb used here:

* **numeric** columns → fill with the **median** (robust: unaffected by extreme values,
  which matters because we are *not* removing outliers in Experiment 1)
* **categorical** columns → fill with the **mode** (most frequent category)

⚠️ **The critical detail:** the median/mode must be computed from the **training folds only**,
never from the whole dataset. Step 6 shows how `Pipeline` guarantees this automatically.


In [ ]:
# Table of missing values for your report
missing_table = pd.DataFrame({
    "missing_count": X.isna().sum(),
    "missing_%": (X.isna().mean()*100).round(2)
}).sort_values("missing_count", ascending=False)
missing_table[missing_table.missing_count > 0]


In [ ]:
# Distributions of the numeric features (Experiment 1 = we only LOOK, we don't remove)
X[NUMERIC].hist(bins=25, figsize=(14, 9), color="#2e86c1", edgecolor="white")
plt.suptitle("Distribution of numeric features (before any outlier treatment)", y=1.00)
plt.tight_layout(); plt.show()


In [ ]:
# Boxplots reveal outliers. We COUNT them now and report them,
# but in Experiment 1 we deliberately leave them in the data.
q1, q3 = X[NUMERIC].quantile(0.25), X[NUMERIC].quantile(0.75)
iqr = q3 - q1
out_counts = (((X[NUMERIC] < q1 - 1.5*iqr) | (X[NUMERIC] > q3 + 1.5*iqr)).sum()
              .sort_values(ascending=False))

desc = X[NUMERIC].describe().T
desc["IQR_outliers"] = out_counts
display(desc.round(2))

fig, axes = plt.subplots(3, 5, figsize=(15, 8))
for axx, col in zip(axes.ravel(), NUMERIC):
    sns.boxplot(y=X[col], ax=axx, color="#5dade2")
    axx.set_title(f"{col}  ({out_counts[col]} outliers)", fontsize=9)
    axx.set_ylabel("")
for axx in axes.ravel()[len(NUMERIC):]:
    axx.axis("off")
plt.suptitle("Boxplots — outliers detected by the 1.5×IQR rule", y=1.00)
plt.tight_layout(); plt.show()


---
## Step 5 — Encoding categorical variables

Machine-learning models need numbers, not the words `"yes"` / `"normal"` / `"present"`.

We use **One-Hot Encoding** with `drop="if_binary"`. All 10 categorical columns here are
binary (exactly 2 categories), so each becomes **one** 0/1 column — e.g. `htn` → `htn_yes`
(1 = hypertension, 0 = no hypertension). No extra redundant column is created.

**Why one-hot and not just `LabelEncoder` (0,1,2,…)?**
For binary variables the two are equivalent. But `LabelEncoder` on a 3+ category column would
invent a fake *order* (`a=0 < b=1 < c=2`) that the model would wrongly treat as meaningful.
One-hot never does this, so it is the safe default.

**`handle_unknown="ignore"`** matters for cross-validation: if a rare category happens to appear
only in the test fold, the encoder outputs all-zeros instead of crashing.

`sg`, `al` and `su` stay **numeric** — they are *graded* clinical readings where the order
genuinely means something (albumin 4 really is worse than albumin 1).


---
## Step 6 — ⭐ The most important cell: a leak-free Pipeline

### What is data leakage?

Leakage is when information from the **test fold** sneaks into how the model was **trained**.
The score then looks great in your notebook and collapses on real patients.

The classic beginner mistake:

```python
# ❌ WRONG — this is leakage
X_filled = X.fillna(X.median())         # median computed from ALL 400 rows
X_scaled = StandardScaler().fit_transform(X_filled)   # mean/SD from ALL 400 rows
cross_val_score(model, X_scaled, y, cv=5)             # test folds already influenced this
```

The median and the scaler's mean/SD were computed using the test-fold rows.
The test fold is no longer truly unseen.

### The fix

Put **every** learned preprocessing step inside a `Pipeline`, and hand the *whole pipeline*
to the cross-validator. Then for each fold scikit-learn calls:

* `.fit()` → **only on the 4 training folds** (learns median, mode, mean, SD, categories)
* `.transform()` → applies those *training* numbers to the held-out fold

The held-out fold contributes nothing to the preprocessing. That is exactly what we want.


In [ ]:
def build_preprocessor(treat_outliers=False):
    """All LEARNED preprocessing lives here, so it is refitted every fold."""

    numeric_steps = [("impute", SimpleImputer(strategy="median"))]
    if treat_outliers:                       # used only in Experiment 2
        numeric_steps.append(("winsorize", IQRWinsorizer(factor=1.5)))
    numeric_steps.append(("scale", StandardScaler()))

    numeric_pipe = Pipeline(numeric_steps)

    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),          # mode
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")),
    ])

    return ColumnTransformer([
        ("num", numeric_pipe,     NUMERIC),
        ("cat", categorical_pipe, CATEGORICAL),
    ])

# quick sanity check on the shape after encoding
_check = build_preprocessor().fit_transform(X, y)
print("Feature matrix after impute + scale + one-hot:", _check.shape)
print("Any NaN left? ->", np.isnan(_check).any())


### Proof that it matters — let's actually measure it

Rather than take the leakage warning on faith, we compare the wrong way against the right way
on this very dataset.


In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
make_clf = lambda: LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)

# ❌ WRONG: preprocessor fitted once on 100% of the data
X_leaky = build_preprocessor().fit(X).transform(X)
wrong_auc = cross_val_score(make_clf(), X_leaky, y, cv=cv, scoring="roc_auc")
wrong_acc = cross_val_score(make_clf(), X_leaky, y, cv=cv, scoring="accuracy")

# ✅ CORRECT: preprocessor is a pipeline step, refitted inside every fold
pipe = Pipeline([("prep", build_preprocessor()), ("clf", make_clf())])
right_auc = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc")
right_acc = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")

print(f"WRONG   (preprocess before CV):  AUC {wrong_auc.mean():.4f}   Acc {wrong_acc.mean():.4f}")
print(f"CORRECT (preprocess inside CV):  AUC {right_auc.mean():.4f}   Acc {right_acc.mean():.4f}")
print(f"\nOptimistic bias: AUC {wrong_auc.mean()-right_auc.mean():+.4f}, "
      f"Accuracy {wrong_acc.mean()-right_acc.mean():+.4f}")


**Interpret the number you just got honestly in your report.**

On this dataset the gap is essentially **zero**. That is *not* evidence that leakage is harmless —
it is evidence that this dataset is unusually forgiving: the classes are almost perfectly
separable, and with 400 rows a fold median is nearly identical to the global median.

Leakage bites hard when the dataset is **small**, the signal is **weak**, or preprocessing is
**aggressive** (feature selection on all data, SMOTE before splitting, target encoding).
The pipeline discipline costs nothing here and protects you there — so we keep it.
This is exactly the kind of point that earns marks in the *Discussion* section.


---
## Step 7 — 5-fold Stratified Cross-Validation

### Why cross-validation instead of one train/test split?
With only 400 patients, a single 80/20 split gives an 80-patient test set. Change the random
seed and your accuracy can swing by several percent. Cross-validation splits the data into
**5 folds**, trains 5 times (each fold held out exactly once), and **every patient is tested
exactly once**. You get a mean *and* a standard deviation — far more trustworthy.

### Why *Stratified*?
The dataset is imbalanced: **250 CKD vs 150 notckd (62.5% / 37.5%)**.
`StratifiedKFold` forces every fold to keep that same 62.5/37.5 ratio.
Plain `KFold` could produce a fold with very few notckd patients, making Specificity
wildly unstable.

`shuffle=True` matters here because the raw CSV is **sorted by class** (all CKD rows first).
Without shuffling, the folds would be catastrophically unbalanced.


In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("Fold composition check (proportions must stay ~62.5% / 37.5%):\n")
for k, (tr, te) in enumerate(cv.split(X, y), 1):
    ytr, yte = y.iloc[tr], y.iloc[te]
    print(f"  Fold {k}: train n={len(tr)} ({ytr.mean()*100:.1f}% ckd) | "
          f"test n={len(te)} ({yte.mean()*100:.1f}% ckd)  "
          f"[ckd={int(yte.sum())}, notckd={int((yte==0).sum())}]")


---
## Step 8 — Which classification algorithm?

There is no single "correct" answer, so we compare **four** standard algorithms and let the
evidence decide. This is much stronger for a project report than picking one arbitrarily.

| Algorithm | Why it is a sensible choice here |
|---|---|
| **Logistic Regression** | The clinical-research standard. Interpretable (coefficients = odds ratios), gives well-behaved probabilities. Needs scaling — our pipeline does it. |
| **Decision Tree** | Human-readable rules a doctor can check. Handles mixed data. Prone to overfitting — a useful, honest baseline. |
| **Random Forest** | Usually the strongest on small clinical tables. Averages many trees → robust to outliers and noise. Gives feature importances. |
| **SVM (RBF kernel)** | Very effective on small, well-separated datasets. `probability=True` lets us draw the ROC curve. |

**Recommendation for your report:** present **Random Forest** as the primary model
(best overall balance, plus interpretable feature importances) and report the other three
as comparisons. If your run shows a different winner, report *that* — never bend results to
a pre-written conclusion.


In [ ]:
def build_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
        "Decision Tree":       DecisionTreeClassifier(random_state=RANDOM_STATE),
        "Random Forest":       RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
        "SVM (RBF)":           SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
    }

for n, m in build_models().items():
    print(f"  {n:22s} -> {type(m).__name__}")


---
## Step 9 — Confusion Matrix, TP / TN / FP / FN, and every metric

### The confusion matrix
With **CKD = 1 = positive**, and forcing `labels=[0, 1]` so the orientation is always the same:

|                       | Predicted **notckd** | Predicted **ckd** |
|-----------------------|----------------------|-------------------|
| **Actual notckd (0)** | **TN**               | **FP**            |
| **Actual ckd (1)**    | **FN**               | **TP**            |

* **TP** — has CKD, correctly flagged as CKD ✅
* **TN** — healthy, correctly cleared ✅
* **FP** — healthy, wrongly flagged as CKD ❌ (*false alarm* → unnecessary worry and tests)
* **FN** — **has CKD but was missed** ❌❌ (*the dangerous error* → untreated kidney disease)

> ⚠️ Always pass `labels=[0, 1]` to `confusion_matrix`. Without it the row/column order
> depends on what happens to be in the data, and students routinely swap Sensitivity
> and Specificity by accident.

### The formulas

$$\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN} \qquad
\textbf{Sensitivity (Recall, TPR)}=\frac{TP}{TP+FN}$$

$$\textbf{Specificity (TNR)}=\frac{TN}{TN+FP} \qquad
\textbf{Precision (PPV)}=\frac{TP}{TP+FP}$$

$$\textbf{F1}=2\cdot\frac{\text{Precision}\times\text{Sensitivity}}{\text{Precision}+\text{Sensitivity}}$$

* **Sensitivity** = of all patients who *really* have CKD, what fraction did we catch?
  **This is the most important metric in a screening test** — a missed CKD case is the costly error.
* **Specificity** = of all *healthy* people, what fraction did we correctly clear?
* **Precision** = when the model says "CKD", how often is it right?
* **F1** = harmonic mean of Precision and Sensitivity — a single balanced number.
  It is the harmonic (not arithmetic) mean, so it stays low if *either* one is poor.


In [ ]:
def metrics_from_cm(y_true, y_pred, y_score):
    """Every requested metric, computed explicitly from TP/TN/FP/FN."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    accuracy    = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan   # recall / TPR
    specificity = tn / (tn + fp) if (tn + fp) else np.nan   # TNR
    precision   = tp / (tp + fp) if (tp + fp) else np.nan   # PPV
    f1 = (2 * precision * sensitivity / (precision + sensitivity)
          if (precision + sensitivity) else np.nan)
    auc_value   = roc_auc_score(y_true, y_score)

    return dict(TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                Accuracy=accuracy, Sensitivity=sensitivity,
                Specificity=specificity, Precision=precision,
                F1=f1, AUC=auc_value)

# --- worked mini-example so you can verify the formulas by hand ---
demo = metrics_from_cm(np.array([1,1,1,1,0,0,0,0]),
                       np.array([1,1,1,0,0,0,0,1]),
                       np.array([.9,.8,.7,.4,.3,.2,.1,.6]))
print("Hand-checkable example: TP=3, FN=1, TN=3, FP=1")
print(f"  Sensitivity = 3/(3+1) = 0.75  -> got {demo['Sensitivity']:.2f}")
print(f"  Specificity = 3/(3+1) = 0.75  -> got {demo['Specificity']:.2f}")
print(f"  Precision   = 3/(3+1) = 0.75  -> got {demo['Precision']:.2f}")
print(f"  F1          = 0.75            -> got {demo['F1']:.2f}")


---
## Step 10 — Run the full cross-validated evaluation

For each model we do the 5 folds manually (instead of `cross_val_score`) because we need
three things back: the per-fold metrics, the **out-of-fold predictions**, and the
**predicted probabilities** for the ROC curve.

**Two ways to report — we produce both:**

1. **Per-fold mean ± SD** — shows how *stable* the model is. Use this in your main results table.
2. **Pooled out-of-fold (OOF)** — every patient predicted exactly once while held out, then one
   single confusion matrix over all 400. This is the honest matrix to *plot*, because adding up
   five separate confusion matrices is exactly equivalent and avoids averaging integers.


In [ ]:
def evaluate(name, model, X, y, treat_outliers=False):
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    per_fold, roc_curves = [], []
    oof_pred  = np.zeros(len(y), dtype=int)
    oof_score = np.zeros(len(y), dtype=float)

    for k, (tr, te) in enumerate(cv.split(X, y), 1):
        pipe = Pipeline([("prep", build_preprocessor(treat_outliers)),
                         ("clf",  model)])

        pipe.fit(X.iloc[tr], y.iloc[tr])       # <-- learns ONLY from training folds
        pred  = pipe.predict(X.iloc[te])
        score = pipe.predict_proba(X.iloc[te])[:, 1]   # P(CKD)

        oof_pred[te], oof_score[te] = pred, score

        m = metrics_from_cm(y.iloc[te], pred, score); m["Fold"] = k
        per_fold.append(m)

        fpr, tpr, _ = roc_curve(y.iloc[te], score)
        roc_curves.append((fpr, tpr, auc(fpr, tpr)))

    fold_df = pd.DataFrame(per_fold).set_index("Fold")
    pooled  = metrics_from_cm(y, oof_pred, oof_score)
    return dict(name=name, fold_df=fold_df, pooled=pooled,
                roc_curves=roc_curves, oof_pred=oof_pred, oof_score=oof_score)


results, summary_rows = {}, []
for name, model in build_models().items():
    r = results[name] = evaluate(name, model, X, y, treat_outliers=False)
    print(f"\n{'='*76}\n{name}\n{'='*76}")
    print(r["fold_df"][["TP","TN","FP","FN","Accuracy","Sensitivity",
                        "Specificity","Precision","F1","AUC"]].round(4).to_string())
    fd = r["fold_df"]
    print("\n  Mean +/- SD across the 5 folds:")
    for m in ["Accuracy","Sensitivity","Specificity","Precision","F1","AUC"]:
        print(f"    {m:12s} {fd[m].mean():.4f} +/- {fd[m].std():.4f}")

    p = r["pooled"]
    summary_rows.append({"Model": name,
        **{k: p[k] for k in ["Accuracy","Sensitivity","Specificity",
                             "Precision","F1","AUC","TP","TN","FP","FN"]}})

summary = pd.DataFrame(summary_rows).set_index("Model")


---
## Step 11 — Results table (this goes straight into your report)


In [ ]:
print("TABLE: Pooled out-of-fold performance, 5-fold Stratified CV (CKD = positive class)\n")
display(summary[["Accuracy","Sensitivity","Specificity","Precision","F1","AUC"]]
        .round(4).style.background_gradient(cmap="Greens", axis=None)
        .format("{:.4f}"))

print("\nTABLE: Pooled confusion-matrix counts (out of all 400 patients)\n")
display(summary[["TP","TN","FP","FN"]])

# Save to CSV so you can paste into Word/LaTeX
summary.round(4).to_csv("results_summary_no_outlier.csv")
print("Saved -> results_summary_no_outlier.csv")


In [ ]:
# Per-fold detail table for the model of your choice (stability evidence)
BEST = summary["AUC"].idxmax()
print(f"Best model by pooled AUC: {BEST}\n")
print(f"TABLE: Per-fold results — {BEST}")
display(results[BEST]["fold_df"][["TP","TN","FP","FN","Accuracy","Sensitivity",
                                  "Specificity","Precision","F1","AUC"]].round(4))
results[BEST]["fold_df"].round(4).to_csv("results_folds_best_model.csv")


---
## Step 12 — Plot the Confusion Matrices


In [ ]:
def plot_confusion(pooled, name, ax):
    cm = np.array([[pooled["TN"], pooled["FP"]],
                   [pooled["FN"], pooled["TP"]]])
    labels = np.array([[f"TN\n{cm[0,0]}", f"FP\n{cm[0,1]}"],
                       [f"FN\n{cm[1,0]}", f"TP\n{cm[1,1]}"]])
    sns.heatmap(cm, annot=labels, fmt="", cmap="Blues", cbar=False,
                square=True, annot_kws={"size": 13}, ax=ax,
                xticklabels=["Pred notckd", "Pred ckd"],
                yticklabels=["True notckd", "True ckd"])
    ax.set_title(name, fontsize=11)

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for axx, (name, r) in zip(axes, results.items()):
    plot_confusion(r["pooled"], name, axx)
plt.suptitle("Confusion Matrices — pooled out-of-fold, 5-fold Stratified CV "
             "(Experiment 1: no outlier removal)", y=1.04, fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# Large single confusion matrix for the best model (use this figure in the report)
fig, ax = plt.subplots(figsize=(5.5, 4.8))
plot_confusion(results[BEST]["pooled"], f"{BEST} — pooled out-of-fold (n=400)", ax)
plt.tight_layout(); plt.show()

p = results[BEST]["pooled"]
print(f"\nReading this matrix for {BEST}:")
print(f"  TP = {p['TP']:3d}  CKD patients correctly identified")
print(f"  TN = {p['TN']:3d}  healthy people correctly cleared")
print(f"  FP = {p['FP']:3d}  healthy people wrongly flagged as CKD (false alarm)")
print(f"  FN = {p['FN']:3d}  CKD patients MISSED  <-- the clinically dangerous error")
print(f"\n  Sensitivity = TP/(TP+FN) = {p['TP']}/({p['TP']}+{p['FN']}) = {p['Sensitivity']:.4f}")
print(f"  Specificity = TN/(TN+FP) = {p['TN']}/({p['TN']}+{p['FP']}) = {p['Specificity']:.4f}")
print(f"  Precision   = TP/(TP+FP) = {p['TP']}/({p['TP']}+{p['FP']}) = {p['Precision']:.4f}")
print(f"  F1          = {p['F1']:.4f}")


---
## Step 13 — ROC curve and AUC

### What is a ROC curve?
The model does not really output "ckd"/"notckd" — it outputs a **probability** of CKD.
To get a label you apply a threshold (default 0.5). The ROC curve asks:
*what if we tried **every** possible threshold?*

For each threshold we plot **Sensitivity (TPR)** on the y-axis against
**1 − Specificity (FPR)** on the x-axis.

* Top-left corner = perfect (catches every patient, no false alarms)
* Diagonal line = random guessing

### What is AUC?
**AUC = Area Under the ROC Curve**, between 0 and 1. Its interpretation is beautifully concrete:

> **AUC is the probability that the model gives a randomly chosen CKD patient a higher
> risk score than a randomly chosen healthy person.**

AUC = 0.5 → useless; AUC = 1.0 → perfect ranking. Crucially, AUC is **threshold-independent**
and far less misleading than accuracy on imbalanced data.

We plot all 5 fold curves plus the **mean ROC** with a ±1 SD band.


In [ ]:
def plot_roc(r, ax):
    mean_fpr, tprs = np.linspace(0, 1, 200), []
    for k, (fpr, tpr, a) in enumerate(r["roc_curves"], 1):
        ax.plot(fpr, tpr, lw=1, alpha=0.45, label=f"Fold {k} (AUC={a:.4f})")
        t = np.interp(mean_fpr, fpr, tpr); t[0] = 0.0
        tprs.append(t)
    mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
    ax.plot(mean_fpr, mean_tpr, "b-", lw=2.4,
            label=f"Mean ROC (AUC={auc(mean_fpr, mean_tpr):.4f})")
    sd = np.std(tprs, axis=0)
    ax.fill_between(mean_fpr, np.maximum(mean_tpr-sd, 0), np.minimum(mean_tpr+sd, 1),
                    color="b", alpha=0.12, label="+/- 1 SD")
    ax.plot([0,1], [0,1], "k--", lw=1, label="Chance (AUC=0.5)")
    ax.set_xlabel("1 - Specificity  (False Positive Rate)")
    ax.set_ylabel("Sensitivity  (True Positive Rate)")
    ax.set_title(r["name"], fontsize=11)
    ax.legend(loc="lower right", fontsize=7.5)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for axx, (name, r) in zip(axes.ravel(), results.items()):
    plot_roc(r, axx)
plt.suptitle("ROC Curves — 5-fold Stratified CV (Experiment 1: no outlier removal)",
             y=1.00, fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# Single combined figure: mean ROC of all four models on one axis
fig, ax = plt.subplots(figsize=(6.5, 5.6))
mean_fpr = np.linspace(0, 1, 200)
for name, r in results.items():
    tprs = []
    for fpr, tpr, _ in r["roc_curves"]:
        t = np.interp(mean_fpr, fpr, tpr); t[0] = 0.0
        tprs.append(t)
    mt = np.mean(tprs, axis=0); mt[-1] = 1.0
    ax.plot(mean_fpr, mt, lw=2,
            label=f"{name} (AUC = {r['pooled']['AUC']:.4f})")
ax.plot([0,1],[0,1],"k--",lw=1,label="Chance (AUC = 0.5)")
ax.set_xlabel("1 - Specificity (FPR)"); ax.set_ylabel("Sensitivity (TPR)")
ax.set_title("ROC comparison of all models\n5-fold Stratified CV, pooled AUC")
ax.legend(loc="lower right", fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# Bar chart comparing every metric across models
metrics = ["Accuracy","Sensitivity","Specificity","Precision","F1","AUC"]
xpos, w = np.arange(len(summary)), 0.13
fig, ax = plt.subplots(figsize=(11, 5))
for i, m in enumerate(metrics):
    bars = ax.bar(xpos + i*w - 2.5*w, summary[m], w, label=m)
    for b, v in zip(bars, summary[m]):
        ax.text(b.get_x()+b.get_width()/2, v+0.001, f"{v:.3f}",
                ha="center", fontsize=6.5, rotation=90)
ax.set_xticks(xpos); ax.set_xticklabels(summary.index, fontsize=10)
ax.set_ylim(0.90, 1.02); ax.set_ylabel("Score (pooled out-of-fold)")
ax.set_title("Model comparison — 5-fold Stratified CV (Experiment 1)")
ax.legend(ncol=6, fontsize=9, loc="lower center"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()


---
## Step 14 — Which features drive the prediction?

Useful for the *Discussion* section: do the model's top features match known CKD medicine?

*(Note: this is fitted on the full dataset for **description only** — it is not a performance
estimate, so there is no leakage concern here.)*


In [ ]:
fi_pipe = Pipeline([("prep", build_preprocessor()),
                    ("clf", RandomForestClassifier(n_estimators=300,
                                                   random_state=RANDOM_STATE))])
fi_pipe.fit(X, y)
names = [n.split("__", 1)[1] for n in fi_pipe.named_steps["prep"].get_feature_names_out()]
imp = pd.Series(fi_pipe.named_steps["clf"].feature_importances_,
                index=names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5.5))
top = imp.head(15)
ax.barh(top.index[::-1], top.values[::-1], color="#2e7d32")
ax.set_xlabel("Gini importance"); ax.set_title("Top 15 most important features (Random Forest)")
plt.tight_layout(); plt.show()

display(imp.head(15).round(4).to_frame("importance"))


---
# EXPERIMENT 2 — with outlier treatment

Your first experiment (everything above) deliberately kept all outliers. Now we repeat it
with outlier treatment so you can compare, exactly as your teacher suggested.

### Why we **cap (winsorise)** instead of **deleting** rows

1. **Clinical reason.** In this dataset the "outliers" are things like serum creatinine = 76
   and blood urea = 391. Those are not typos — they are **the sickest kidney patients**.
   Deleting them means deleting the very cases a CKD detector exists to find.
2. **Methodological reason.** Outlier *removal* changes the number of rows, so different
   experiments would be scored on different test sets and the comparison becomes meaningless.
   Capping keeps all 400 patients and keeps the comparison fair.
3. **Leakage reason.** The IQR fences (Q1 − 1.5·IQR, Q3 + 1.5·IQR) are **statistics learned
   from data**. So, exactly like the median and the scaler, they must be computed on the
   **training folds only**. We implement it as a scikit-learn transformer so the pipeline
   handles this automatically.

> If your teacher specifically wants *row removal*, do it on the **training fold only**,
> never on the test fold — otherwise you are quietly deleting the hard cases from your exam.


In [ ]:
class IQRWinsorizer(BaseEstimator, TransformerMixin):
    """Cap numeric values at IQR fences LEARNED FROM TRAINING DATA ONLY.

    Because this is a transformer inside the Pipeline, .fit() only ever sees
    the 4 training folds. The held-out fold is merely .transform()-ed using
    those fences -> no leakage.
    """
    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = q1 - self.factor * iqr
        self.upper_ = q3 + self.factor * iqr
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        return np.clip(X, self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)   # columns unchanged

print("IQRWinsorizer ready.")


In [ ]:
results2, rows2 = {}, []
for name, model in build_models().items():
    r = results2[name] = evaluate(name, model, X, y, treat_outliers=True)  # <-- capping ON
    p = r["pooled"]
    rows2.append({"Model": name,
        **{k: p[k] for k in ["Accuracy","Sensitivity","Specificity",
                             "Precision","F1","AUC","TP","TN","FP","FN"]}})

summary2 = pd.DataFrame(rows2).set_index("Model")
print("EXPERIMENT 2 — with IQR capping (pooled out-of-fold)\n")
display(summary2[["Accuracy","Sensitivity","Specificity","Precision","F1","AUC"]].round(4))
summary2.round(4).to_csv("results_summary_outlier_capped.csv")


In [ ]:
# Side-by-side comparison table for the report
cmp = pd.concat({
    "Exp 1: no outlier removal": summary[["Accuracy","Sensitivity","Specificity",
                                          "Precision","F1","AUC"]],
    "Exp 2: IQR capped":         summary2[["Accuracy","Sensitivity","Specificity",
                                           "Precision","F1","AUC"]],
}, axis=1).round(4)
print("TABLE: Experiment 1 vs Experiment 2\n")
display(cmp)

print("\nChange (Exp 2 - Exp 1); positive = capping helped:\n")
delta = (summary2[["Accuracy","Sensitivity","Specificity","Precision","F1","AUC"]]
         - summary[["Accuracy","Sensitivity","Specificity","Precision","F1","AUC"]])
display(delta.round(4).style.background_gradient(cmap="RdYlGn", axis=None).format("{:+.4f}"))
cmp.to_csv("results_experiment_comparison.csv")


In [ ]:
# Visual comparison of the two experiments
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True)
for axx, (sm, title) in zip(axes, [(summary,  "Experiment 1: no outlier removal"),
                                   (summary2, "Experiment 2: IQR capping")]):
    xpos, w = np.arange(len(sm)), 0.15
    for i, m in enumerate(["Accuracy","Sensitivity","Specificity","F1","AUC"]):
        axx.bar(xpos + i*w - 2*w, sm[m], w, label=m)
    axx.set_xticks(xpos); axx.set_xticklabels(sm.index, fontsize=8, rotation=12)
    axx.set_title(title); axx.set_ylim(0.90, 1.01); axx.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Score"); axes[0].legend(ncol=5, fontsize=8, loc="lower center")
plt.tight_layout(); plt.show()


---
## Step 15 — How to write up your project

### Methodology (write it in this order)

1. **Dataset.** UCI Chronic Kidney Disease dataset, 400 patient records, 24 clinical
   attributes (11 numeric, 13 categorical after removing `id`) and a binary target
   `classification` ∈ {ckd, notckd}. Class distribution: **250 CKD (62.5%) / 150 notckd (37.5%)**.
2. **Data cleaning.** Report the four concrete defects you found and fixed: the `'ckd\t'`
   label, the `'\tno'`/`'\tyes'`/`' yes'` variants in `dm` and `cad`, and the `'\t?'`
   placeholders that forced `pcv`/`wc`/`rc` to be read as text.
3. **Missing values.** State the percentage missing per column (your table from Step 4) and
   note that only a minority of rows are complete, so listwise deletion was rejected.
   Numeric → median imputation; categorical → mode imputation.
4. **Encoding.** One-hot encoding with `drop="if_binary"` for the 10 binary categorical
   variables; `sg`, `al`, `su` retained as ordered numeric.
5. **Scaling.** `StandardScaler` (zero mean, unit variance) — required by Logistic
   Regression and SVM.
6. **Leakage control.** *This is the paragraph that gets you marks.* State that all
   imputation, scaling, encoding (and, in Experiment 2, the IQR fences) were embedded in a
   `sklearn.pipeline.Pipeline` and refitted **within each training fold**, so no information
   from a held-out fold ever influenced preprocessing.
7. **Validation.** 5-fold **Stratified** cross-validation, `shuffle=True`,
   `random_state=42`. Justify stratification with the 62.5/37.5 imbalance and note that
   the raw file is sorted by class, making shuffling essential.
8. **Models.** Logistic Regression, Decision Tree, Random Forest (300 trees), SVM (RBF).
9. **Metrics.** Define CKD as positive, then give the TP/TN/FP/FN table and the formulas
   for Accuracy, Sensitivity, Specificity, Precision, F1 and AUC.

### Results
Present, in this order: the missing-value table → class-distribution figure → the main
results table (Step 11) → confusion matrices (Step 12) → ROC curves (Step 13) →
feature importance (Step 14) → the Experiment 1 vs 2 comparison (Step 15's table).
Report both **pooled out-of-fold** values and **per-fold mean ± SD**.

### Discussion — points that will genuinely impress
* **Sensitivity vs Specificity trade-off.** In screening, a **false negative (FN)** — a missed
  CKD patient — is far costlier than a false alarm. So judge models primarily on
  **Sensitivity**, and mention you could lower the decision threshold below 0.5 to trade
  a little Specificity for higher Sensitivity.
* **Why the scores are so high.** Be honest about this. `hemo`, `pcv`, `sg`, `sc` and `rc`
  are the *defining diagnostic criteria* of CKD, so the classes are almost linearly separable.
  This dataset is a clean teaching benchmark, **not** a realistic screening challenge, and
  ~99% accuracy here would **not** transfer to raw hospital data.
* **The leakage experiment.** Report the measured gap from Step 6. If it was ~0, explain
  *why*: near-perfect separability plus n=400 means fold medians ≈ global medians. Then note
  that leakage does real damage on small/noisy datasets, so the pipeline discipline is
  cheap insurance, not wasted effort.
* **Missingness is informative.** `rbc` is ~38% missing. Missingness is probably not random
  (sicker patients get more tests), so imputing it with the mode may itself destroy signal.
  Suggest adding missing-indicator features as future work.
* **Clinical plausibility.** Check whether your top features (likely `hemo`, `pcv`, `sc`,
  `sg`, `rc`, `htn`) match known CKD markers — anaemia and impaired filtration. They should.
* **Limitations.** Only 400 patients from a single source; no external validation cohort;
  hyperparameters left at defaults (a nested CV would be needed to tune them without bias);
  results are cross-validated estimates, not a prospective clinical trial.

### Conclusion
Summarise: the best model, its Sensitivity / Specificity / F1 / AUC, the fact that the
evaluation was leak-free and stratified, and one honest sentence on limitations plus one
line of future work (external validation, threshold tuning, missing-indicator features,
nested CV for hyperparameter tuning).


---
## Appendix — Save every figure and table for your report


In [ ]:
import os
os.makedirs("report_outputs", exist_ok=True)

# tables
summary.round(4).to_csv("report_outputs/table1_experiment1_summary.csv")
summary2.round(4).to_csv("report_outputs/table2_experiment2_summary.csv")
cmp.to_csv("report_outputs/table3_experiment_comparison.csv")
missing_table.to_csv("report_outputs/table4_missing_values.csv")
imp.round(4).to_frame("importance").to_csv("report_outputs/table5_feature_importance.csv")
for name, r in results.items():
    fn = name.lower().replace(" ", "_").replace("(","").replace(")","")
    r["fold_df"].round(4).to_csv(f"report_outputs/folds_exp1_{fn}.csv")

print("Saved tables:")
for f in sorted(os.listdir("report_outputs")):
    print("  report_outputs/" + f)

# In Colab, download everything as a zip:
#   !zip -r report_outputs.zip report_outputs
#   from google.colab import files; files.download("report_outputs.zip")
